# Instanovo-Glyco Training Data Analysis

In [2]:
import os
import sys
import glob
import logging
import itertools
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Fix this later, imports should work without this
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), os.pardir)))


from common.utils import collect_files, get_or_create_folder, load_ipc_files
from common.logger import get_logger_config
from common.constants import (
    BASE_RAW_DATA_DIR,
    BASE_LOGS_DIR,
    BASE_PLOTS_DIR,
    BASE_REPORTS_CSV_DIR,
)

In [3]:
target_data = "PXD025859"  # PXD035158
artifacts_sub_dir = (BASE_RAW_DATA_DIR / target_data).as_posix().split("/")[-1]
logs_dir = get_or_create_folder(BASE_LOGS_DIR / artifacts_sub_dir)
plots_dir = get_or_create_folder(BASE_PLOTS_DIR / artifacts_sub_dir)
csv_dir = get_or_create_folder(BASE_REPORTS_CSV_DIR / artifacts_sub_dir)

In [4]:
logger_config = get_logger_config(subdir=artifacts_sub_dir)
logging.config.dictConfig(logger_config)
logger = logging.getLogger(__name__)
logging.warning(
    f"Changing matplotlib font from {plt.rcParams['font.family']} to ['monospace']"
)
plt.rcParams["font.family"] = ["monospace"]

2025-04-13 21:23:12,465 - root - WARNING - Changing matplotlib font from ['sans-serif'] to ['monospace']


In [4]:
logger.info(
    f"Starting with artifacts subdir set to {artifacts_sub_dir}, log_dir set to {logs_dir} and plots_dir set to {plots_dir}"
)

2025-04-13 20:34:05,417 - __main__ - INFO - Starting with artifacts subdir set to PXD025859, log_dir set to /home/hjisaac/AI4Science/instanovo_instadeep/InstanovoGlyco/reports/logs/PXD025859 and plots_dir set to /home/hjisaac/AI4Science/instanovo_instadeep/InstanovoGlyco/reports/plots/PXD025859


In [6]:
# Grab all ipc files of interest but ATTENTION;
# loading all many ipc files will increase the computation time
ipc_files = collect_files(BASE_RAW_DATA_DIR / target_data)
logger.info(f"Found {len(ipc_files)} IPC files")
df = load_ipc_files(ipc_files, verbose=True)
df.head(20)

2025-04-13 16:30:26,780 - __main__ - INFO - Found 52 IPC files



Processing file 0: /home/hjisaac/AI4Science/instanovo_instadeep/InstanovoGlyco/data/raw/PXD025859/ShenJ_FourStandardGlycoproteins_DeSialy_CE20_33_Run1.ipc
File 0 : /home/hjisaac/AI4Science/instanovo_instadeep/InstanovoGlyco/data/raw/PXD025859/ShenJ_FourStandardGlycoproteins_DeSialy_CE20_33_Run1.ipc loaded
First 2 rows:
   index                                         scan  \
0     24  controllerType=0 controllerNumber=1 scan=25   
1     43  controllerType=0 controllerNumber=1 scan=44   

                                              header        rt frag_type  \
0  FTMS + c NSI d Full ms2 777.6844@hcd33.00 [120...  3.716610       HCD   
1  FTMS + c NSI d Full ms2 918.1105@hcd33.00 [120...  6.618322       HCD   

   collision_energy  precursor_mz  precursor_charge  precursor_intensity  \
0              33.0    777.348511                 3         74650.296875   
1              33.0    917.777710                 3         50939.238281   

   lower_offset  ...  peptide_observed_mz  pepti

KeyboardInterrupt: 

In [ ]:
assert False, "Raised intentionally"

In [4]:
# Graphing functions go here
def save_figure(filename, save_dir=plots_dir):
    """
    Save the current figure to a file.

    Args:
    filename (str): The name of the file to save the figure to.
    save_dir (str): The directory to save the figure in. Default to "../reports/plots/{artifacts_sub_dir}".
    """

    extensions = [".png", ".jpg", ".jpeg", ".pdf"]
    name, extension = os.path.splitext(filename)
    # We are defaulting the extension to "pdf"
    extension = extension.lower() if extension else ".pdf"

    if extension not in extensions:
        raise ValueError(
            f"Unknown extension {extension} from {filename}. Please choose one from {extensions}"
        )

    save_path = os.path.join(save_dir, f"{name}{extension}")
    plt.savefig(save_path, format=extension[1:], bbox_inches="tight")
    logger.info(f"Saved plot to {save_path}")


def plot_x_y(
    df,
    index,
    x_column,
    y_column,
    x_label=None,
    y_label=None,
    title=None,
    filename=None,
    save=True,
):
    """
    Plot x and y arrays for a given line in the DataFrame using vertical lines.

    Args:
    df (pd.DataFrame): The DataFrame containing the data.
    index (int): The index of the line to plot.
    x_column (str): The name of the x column.
    y_column (str): The name of the y column.
    x_label (str): The label for the x-axis. If None, the x_column name is used.
    y_label (str): The label for the y-axis. If None, the y_column name is used.
    title (str): The title of the plot. If None, a default title is used.
    filename (str): The filename to save the plot. If None, the plot is not saved.
    """
    x_values = df.at[index, x_column]
    y_values = df.at[index, y_column]
    plt.figure(figsize=(10, 6))
    plt.vlines(x_values, ymin=0, ymax=y_values, color="b", alpha=0.7)
    plt.scatter(x_values, y_values, color="b")
    x_label = x_label if x_label else x_column
    y_label = y_label if y_label else y_column
    title = title if title else f"{x_label} vs. {y_label} for index {index}"
    plt.title(title)
    plt.xlabel(x_label)
    plt.ylabel(y_label)
    plt.grid(True, alpha=0.3)
    if save:
        filename = filename or f"{x_column}_vs_{y_column}_for_index_{index}_xy_plot"
        save_figure(filename)
    plt.show()


def plot_quantitative(
    df, column, xlabel=None, ylabel="Frequency", title=None, filename=None, save=True
):
    """
    Plot histogram and KDE for a quantitative (numeric) column.

    Args:
    df (pd.DataFrame): The DataFrame containing the data.
    column (str): The column to plot.
    label (str): The label to display on the plot. If None, the column name is used.
    """
    xlabel = xlabel if xlabel else column
    title = title if title else f"Histogram and KDE for {xlabel.lower()}"
    plt.figure(figsize=(10, 6))
    sns.histplot(df[column], kde=True)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.grid(True, alpha=0.2)
    if save:
        filename = filename or (column + "_histogram")
        save_figure(filename=filename)
    plt.show()


def plot_qualitative(
    df,
    column,
    xlabel="Count",
    ylabel=None,
    title=None,
    top_n=20,
    filename=None,
    save=True,
):
    """
    Plot bar plot for a qualitative (categorical) column.

    Args:
    df (pd.DataFrame): The DataFrame containing the data.
    column (str): The column to plot.
    label (str): The label to display on the plot. If None, the column name is used.
    top_n (int): The number of top values to display.
    """
    ylabel = ylabel if ylabel else column
    title = (
        title
        if title
        else (
            f"Top {top_n} most frequent values for {ylabel.lower()}"
            if top_n
            else ylabel.lower()
        )
    )
    top_values = df[column].value_counts().nlargest(top_n)
    plt.figure(figsize=(10, 6))
    sns.barplot(y=top_values.index, x=top_values.values)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.grid(True, alpha=0.2)
    if save:
        filename = filename or (column + "_barplot")
        save_figure(filename=filename)
    plt.show()


def plot_x_y_bar(
    df,
    column,
    count_column_name,
    xlabel="Duplicate count",
    ylabel="Column combinations",
    title="Duplicate counts for different column combinations",
    xtick_rotation=45,
    filename=None,
    save=True,
):
    """
    Generates a horizontal bar plot for duplicate counts of column combinations.

    Parameters:
    - results_df: pandas DataFrame containing the data.
    - column_name: str, the column name for the combinations to be plotted.
    - count_name: str, the column name for the duplicate counts to be plotted.
    - xlabel: str, label for the x-axis (default is "Duplicate Count").
    - ylabel: str, label for the y-axis (default is "Column Combinations").
    - title: str, the title of the plot (default is "Duplicate Counts for Different Column Combinations").
    - xtick_rotation: int, angle to rotate x-axis ticks (default is 45).
    """
    plt.figure(
        figsize=(12, max(6, len(df) * 0.3))
    )  # Adjust height dynamically based on the number of rows
    plt.barh(
        df[column],
        df[count_column_name],
    )
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title)
    plt.xticks(rotation=xtick_rotation)
    plt.tight_layout()  # Prevent overlapping
    plt.grid(True, alpha=0.4)
    plt.gca().invert_yaxis()
    if save:
        filename = filename or (column + "_based_duplicates_count_barplot")
        save_figure(filename=filename)
    plt.show()

## Columns description

| **Field**               | **Description**                                                                                   |
|--------------------------|---------------------------------------------------------------------------------------------------|
| `index`                 | A unique identifier for each entry in the dataset.                                                |
| `scan`                  | The scan number corresponding to the acquisition of the mass spectrum.                            |
| `header`                | Additional metadata about the scan, such as file source or retention time.                        |
| `rt` (Retention Time)   | The time it takes for the peptide to travel through the chromatographic column.                   |
| `frag_type`             | The type of fragmentation used (e.g., CID, HCD, ETD).                                             |
| `collision_energy`      | The energy applied to fragment precursor ions.                                                    |
| `precursor_mz`          | The mass-to-charge ratio (\(m/z\)) of the precursor ion before fragmentation.                     |
| `precursor_charge`      | The charge state of the precursor ion (e.g., +2, +3).                                             |
| `precursor_intensity`   | The intensity of the precursor ion signal, reflecting its abundance.                              |
| `lower_offset`          | The lower bound of the \(m/z\) window for isolating the precursor ion.                            |
| `upper_offset`          | The upper bound of the \(m/z\) window for isolating the precursor ion.                            |
| `isolation_target`      | The target \(m/z\) value for isolating the precursor ion.                                         |
| `mz`                   | The mass-to-charge ratio (\(m/z\)) of ions detected in the spectrum.                              |
| `intensity`             | The intensity of ions detected in the spectrum, reflecting their abundance.                       |
| `scale_factor`          | A scaling factor applied to intensities for normalization.                                        |
| `peptide`               | The sequence of the identified peptide, without modifications.                                    |
| `modified_peptide`      | The sequence of the peptide with post-translational modifications (e.g., glycosylation).          |
| `peptide_observed_mz`   | The observed \(m/z\) of the peptide in the spectrum.                                              |
| `peptide_calc_mz`       | The theoretical \(m/z\) of the peptide based on its sequence and modifications.                   |
| `delta_mass`            | The difference between observed and calculated mass, indicating potential modifications or errors. |
| `retention`             | The retention time of the peptide, used for identification confirmation.                          |
| `expectation`           | A statistical value (e.g., E-value) indicating the reliability of peptide identification.          |
| `hyperscore`            | A confidence score for peptide identification (e.g., from Mascot or Sequest).                     |
| `nextscore`             | An additional confidence score for peptide identification.                                        |
| `probability`           | The probability that the peptide identification is correct.                                       |
| `auc_intensity`         | The area under the curve (AUC) of the signal intensity, used for quantification.                  |
| `protein`               | The protein to which the peptide belongs, identified from a database.                             |

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800137 entries, 0 to 800136
Data columns (total 27 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   index                800137 non-null  int64  
 1   scan                 800137 non-null  object 
 2   header               800137 non-null  object 
 3   rt                   800137 non-null  float64
 4   frag_type            800137 non-null  object 
 5   collision_energy     800137 non-null  float64
 6   precursor_mz         800137 non-null  float64
 7   precursor_charge     800137 non-null  int64  
 8   precursor_intensity  800137 non-null  float64
 9   lower_offset         800137 non-null  float64
 10  upper_offset         800137 non-null  float64
 11  isolation_target     632951 non-null  float64
 12  mz                   800137 non-null  object 
 13  intensity            800137 non-null  object 
 14  scale_factor         800137 non-null  float32
 15  peptide          

In [ ]:
df[["mz"]].iloc[0].mz

In [11]:
highly_relevant_columns = [
    "peptide",
    "modified_peptide",
    "precursor_mz",
    "precursor_charge",
    "mz",
    "intensity",
    "rt",
    "delta_mass",
    "protein",
]

moderatly_relevant_columns = [
    "scan",
    "header",
    "frag_type",
    "collision_energy",
    "precursor_intensity",
    "peptide_observed_mz",
    "peptide_calc_mz",
    "auc_intensity",
]

less_relevant_columns = [
    "index",
    "lower_offset",
    "upper_offset",
    "isolation_target",
    "scale_factor",
    "expectation",
    "hyperscore",
    "nextscore",
]

### Recap statistics for relevant and less relevant columns

In [6]:
df_described = df[highly_relevant_columns].describe()
df_described.to_csv(csv_dir / "highly_relevant_columns_described_df.csv", index=False)
df_described

NameError: name 'highly_relevant_columns' is not defined

In [ ]:
df_described = df[moderatly_relevant_columns].describe()
df_described.to_csv(
    csv_dir / "moderatly_relevant_columns_described_df.csv", index=False
)
df_described

In [ ]:
df_described = df[less_relevant_columns].describe()
df_described.to_csv(csv_dir / "less_relevant_columns_described_df.csv", index=False)
df_described

### Duplicate investigation

In [ ]:
# Save unique peptides as a single-column CSV
pd.DataFrame({"Unique Peptides": df["peptide"].unique()}).to_csv(
    csv_dir / "unique_peptides.csv", index=False
)

# Save unique modified peptides as a single-column CSV
pd.DataFrame({"Unique Modified Peptides": df["modified_peptide"].unique()}).to_csv(
    csv_dir / "unique_modified_peptides.csv", index=False
)

In [ ]:
# Investigate duplicates
# assert False, "This code block may take minutes to complete; Do you really want to run this code?, If yes, then disable this assertion."
logger.info(
    "Start replacing list or np.ndarray with tuples for internal comparison purposes"
)


def list_to_tuple_func(value):
    logger.info(f"Transforming {value[:3]} to tuple")
    if isinstance(value, (np.ndarray, list)):
        return tuple(value)
    return value


for column in ("mz", "intensity"):

    # Convert array-like values in the specified columns to tuples. Using
    # another new column will make use of a lot of memory. So, let's just
    # overwrite the values in the specified column.
    logger.info(f"Start list replacement for column {column}")
    df[column] = df[column].apply(list_to_tuple_func)
logger.info("Finish replacing list or np.ndarray with tuples")
# List of columns to consider
columns_to_check = [
    "peptide",
    "modified_peptide",
    "precursor_mz",
    "precursor_charge",
    "mz",
    "intensity",
    # "delta_mass",
    # "protein",
]


# Function to generate aligned labels
def format_label(columns_in_combination):
    formatted = []
    for col in columns_to_check:
        if col in columns_in_combination:
            formatted.append(col.ljust(len(col)))  # Keep column name
        else:
            formatted.append(" " * len(col))  # Add spaces of equal width
    return ", ".join(formatted)  # Use separator for clarity


# Initialize a list to store results
results = []

# Iterate through each combination of column sizes (1-combinaison, 2-combinaison, etc.)
for size in range(1, len(columns_to_check) + 1):
    for comb in itertools.combinations(columns_to_check, size):
        # Count duplicates for the current combination of columns
        duplicate_count = df[list(comb)].duplicated().sum()
        # FIXME: Here .debug should be used
        logger.info(
            f"Combination size: {size}, duplicate count: {duplicate_count}, combinations: {comb}"
        )
        # Store the result as a tuple of (combination, duplicate_count)
        results.append(
            {"columns": format_label(comb), "duplicate_count": duplicate_count}
        )

# Convert results into a DataFrame
results_df = pd.DataFrame(results)

# Print the DataFrame with duplicate counts
print(results_df)

# Plotting the results
plot_x_y_bar(
    results_df,
    "columns",
    "duplicate_count",
    xlabel="Count of duplicates",
    ylabel="Column combinations",
    title="Duplicate counts for different column combinations",
    xtick_rotation=45,
)

In [ ]:
# Access the mz and intensity of the most abundant peptide and modification
pd.DataFrame({"Unique Peptides": df["peptide"].unique()}).to_csv(
    csv_dir / "unique_peptides.csv", index=False
)

# Save unique modified peptides as a single-column CSV
pd.DataFrame({"Unique Modified Peptides": df["modified_peptide"].unique()}).to_csv(
    csv_dir / "unique_modified_peptides.csv", index=False
)
# most_abundant_rows.head()

### We want to see from which proteins do those duplicated peptides come

In [7]:
top_peptides = df["peptide"].value_counts().nlargest(20).index

# Filter the original DataFrame to only include those peptides
filtered_df = df[df["peptide"].isin(top_peptides)]

# Group by peptide and list all associated proteins
peptide_to_proteins = (
    filtered_df.groupby("peptide")["protein"]
    .apply(lambda proteins: list(set(proteins)))  # optional: remove duplicates
    .reset_index()
)

In [8]:
filtered_df[["peptide", "protein"]].value_counts()

peptide                                  protein              
MVSHHNLTTGATLINEQWLLTTAK                 sp|P00738|HPT_HUMAN      5608
TPEVTCVVVDVSHEDPEVQFNWYVDGVEVHNAK        sp|P01859|IGHG2_HUMAN    4288
EEQYNSTYR                                sp|P01857|IGHG1_HUMAN    3623
GILGYTEDQVVSCDFNSNSHSSTFDAGAGIALNDNFVK   sp|P16858|G3P_MOUSE      3618
EFNAETFTFHADICTLSEK                      sp|P02768|ALBU_HUMAN     3361
VSDTVVEPYNATLSVHQLVENTDETYSIDNEALYDICFR  sp|Q7TMM9|TBB2A_MOUSE    3136
VDNALQSGNSQESVTEQDSK                     sp|P01834|IGKC_HUMAN     3060
LSFTSVGSITSGYSQSSQVFGR                   sp|P08551|NFL_MOUSE      2712
SLGNVNFTVSAEALESQELCGTEVPSVPEHGR         sp|P01023|A2MG_HUMAN     2675
EEQFNSTFR                                sp|P01859|IGHG2_HUMAN    2655
TENLDVIVNISDTESWGQHVQK                   sp|P14231|AT1B2_MOUSE    2530
VSDTVVEPYNATLSVHQLVENTDETYCIDNEALYDICFR  sp|Q9D6F9|TBB4A_MOUSE    2329
QTFIGHESDINAVAFFPNGYAFTTGSDDATCR         sp|P62880|GBB2_MOUSE     2248
TPEVTCVVVDVSQE

In [13]:
(df["protein"] == "").sum()

np.int64(0)

Index(['MVSHHNLTTGATLINEQWLLTTAK', 'TPEVTCVVVDVSHEDPEVQFNWYVDGVEVHNAK',
       'EEQYNSTYR', 'GILGYTEDQVVSCDFNSNSHSSTFDAGAGIALNDNFVK',
       'AFVHWYVGEGMEEGEFSEAR', 'EFNAETFTFHADICTLSEK',
       'VSDTVVEPYNATLSVHQLVENTDETYSIDNEALYDICFR', 'VDNALQSGNSQESVTEQDSK',
       'LSFTSVGSITSGYSQSSQVFGR', 'SLGNVNFTVSAEALESQELCGTEVPSVPEHGR',
       'EEQFNSTFR', 'TENLDVIVNISDTESWGQHVQK',
       'VSDTVVEPYNATLSVHQLVENTDETYCIDNEALYDICFR',
       'QTFIGHESDINAVAFFPNGYAFTTGSDDATCR', 'TPEVTCVVVDVSQEDPEVQFNWYVDGVEVHNAK',
       'TPEVTCVVVDVSHEDPEVK', 'LSLHRPALEDLLLGSEANLTCTLTGLR',
       'VVLHPNYSQVDIGLIK', 'YLGNATAIFFLPDEGK', 'AYHEQLSVAEITNACFEPANQMVK'],
      dtype='object', name='peptide')

In [18]:
print(peptide_to_proteins)

                                    peptide  \
0                      AFVHWYVGEGMEEGEFSEAR   
1                  AYHEQLSVAEITNACFEPANQMVK   
2                                 EEQFNSTFR   
3                                 EEQYNSTYR   
4                       EFNAETFTFHADICTLSEK   
5    GILGYTEDQVVSCDFNSNSHSSTFDAGAGIALNDNFVK   
6                    LSFTSVGSITSGYSQSSQVFGR   
7               LSLHRPALEDLLLGSEANLTCTLTGLR   
8                  MVSHHNLTTGATLINEQWLLTTAK   
9          QTFIGHESDINAVAFFPNGYAFTTGSDDATCR   
10         SLGNVNFTVSAEALESQELCGTEVPSVPEHGR   
11                   TENLDVIVNISDTESWGQHVQK   
12                      TPEVTCVVVDVSHEDPEVK   
13        TPEVTCVVVDVSHEDPEVQFNWYVDGVEVHNAK   
14        TPEVTCVVVDVSQEDPEVQFNWYVDGVEVHNAK   
15                     VDNALQSGNSQESVTEQDSK   
16  VSDTVVEPYNATLSVHQLVENTDETYCIDNEALYDICFR   
17  VSDTVVEPYNATLSVHQLVENTDETYSIDNEALYDICFR   
18                         VVLHPNYSQVDIGLIK   
19                         YLGNATAIFFLPDEGK   

            

In [ ]:
plot_qualitative(df, "modified_peptide", "Modified peptides")
plot_qualitative(df, "peptide", "Peptide")
plot_qualitative(df, "protein", "Proteins")

In [ ]:
plot_quantitative(df, "precursor_mz", xlabel="Precursor m/z")
plot_quantitative(df, "precursor_charge", xlabel="Precursor charge")
plot_quantitative(df, "delta_mass", xlabel="Delta mass")

In [ ]:
peptide_index = 0
plot_x_y(
    df,
    peptide_index,
    "mz",
    "intensity",
    "m/z",
    "Intensity",
    title=f'm/z vs. Intensity for peptide {df.iloc[peptide_index]["peptide"]}',
)

### Do we have entries that come with modified_peptide set to peptide without any actual modifications ?

In [5]:
fake_modification_df = df[df["peptide"] == df["modified_peptide"]]

In [ ]:
# fake_modification_df.head(6)

In [ ]:
# fake_modification_df[["peptide", "modified_peptide"]]

## Deep missing column/values investigation (Independent section/cells)

In [5]:
projects_dirs = glob.glob(f"{BASE_RAW_DATA_DIR}/*/")
assert projects_dirs, projects_dirs
dirs_to_ignore = ["PXD044641_PXD035158"] 

project_summary = {
}
for project_dir in projects_dirs:
    project_name = project_dir.split("/")[-2]
    if project_name in dirs_to_ignore:
        logger.info(f"Skipping project {project_name} as part of projects to ignore")
        continue
    project_file_paths = collect_files(location=project_dir, ext="ipc")
    _, summary = load_ipc_files(project_file_paths)
    project_summary[project_name] = summary
    
    



Processing files:  26%|██▌       | 7/27 [00:01<00:04,  4.47it/s]2025-04-13 21:23:33,566 - common.utils - INFO - Processing file 7: /home/hjisaac/AI4Science/instanovo_instadeep/InstanovoGlyco/data/raw/PXD026629/YangLuJie-LPS8h-2.ipc
2025-04-13 21:23:33,655 - common.utils - INFO - Processing file 8: /home/hjisaac/AI4Science/instanovo_instadeep/InstanovoGlyco/data/raw/PXD026629/20180831YLJ-H0h-01.ipc
Processing files:  44%|████▍     | 12/27 [00:02<00:02,  7.39it/s]2025-04-13 21:23:34,110 - common.utils - INFO - Processing file 12: /home/hjisaac/AI4Science/instanovo_instadeep/InstanovoGlyco/data/raw/PXD026629/YangLuJie-LPS4h-3.ipc
2025-04-13 21:23:34,193 - common.utils - INFO - Processing file 13: /home/hjisaac/AI4Science/instanovo_instadeep/InstanovoGlyco/data/raw/PXD026629/20180831YLJ-H0h-02.ipc
Processing files:  52%|█████▏    | 14/27 [00:02<00:01,  8.90it/s]2025-04-13 21:23:34,277 - common.utils - INFO - Processing file 14: /home/hjisaac/AI4Science/instanovo_instadeep/InstanovoGlyco/da

In [6]:
project_summary

{'PXD026629': [{'file_path': '/home/hjisaac/AI4Science/instanovo_instadeep/InstanovoGlyco/data/raw/PXD026629/20180904YLJ-VSV4h-02.ipc',
   'row_count': 13878,
   'column_count': 27,
   'columns': ['index',
    'scan',
    'header',
    'rt',
    'frag_type',
    'collision_energy',
    'precursor_mz',
    'precursor_charge',
    'precursor_intensity',
    'lower_offset',
    'upper_offset',
    'isolation_target',
    'mz',
    'intensity',
    'scale_factor',
    'peptide',
    'modified_peptide',
    'peptide_observed_mz',
    'peptide_calc_mz',
    'delta_mass',
    'retention',
    'expectation',
    'hyperscore',
    'nextscore',
    'probability',
    'auc_intensity',
    'protein'],
   'missing_values': {'isolation_target': 2792, 'modified_peptide': 1908}},
  {'file_path': '/home/hjisaac/AI4Science/instanovo_instadeep/InstanovoGlyco/data/raw/PXD026629/20180904YLJ-VSV0h-03.ipc',
   'row_count': 14151,
   'column_count': 27,
   'columns': ['index',
    'scan',
    'header',
    'r

In [9]:
table_rows = []

for project_id, file_info_list in project_summary.items():
    for file_info in file_info_list:
        file_path = file_info['file_path']
        file_name = file_path.split('/')[-1]  # Extract the file name from the path
        row_count = file_info['row_count']
        column_count = file_info['column_count']
        columns = file_info['columns']
        missing_isolation_target = file_info['missing_values']['isolation_target']
        missing_modified_peptide = file_info['missing_values']['modified_peptide']
        
        # Calculate percentages
        percent_missing_isolation_target = (missing_isolation_target / row_count) * 100
        percent_missing_modified_peptide = (missing_modified_peptide / row_count) * 100
        
        table_rows.append({
            'Project ID': project_id,
            'File Name': file_name,
            'Row Count': row_count,
            'Column Count': column_count,
            'Columns': columns,
            'Missing isolation_target': missing_isolation_target,
            'Missing modified_peptide': missing_modified_peptide,
            '% Missing isolation_target': round(percent_missing_isolation_target, 2),
            '% Missing modified_peptide': round(percent_missing_modified_peptide, 2)
        })

In [10]:
df_missing_summary = pd.DataFrame(table_rows)

df_missing_summary.to_csv(BASE_REPORTS_CSV_DIR / "deep_missing_values_investigation_summary_report.csv", index=False)


In [11]:
df_missing_summary.head(10)

,Project ID,File Name,Row Count,Column Count,Columns,Missing isolation_target,Missing modified_peptide,% Missing isolation_target,% Missing modified_peptide
0,PXD026629,20180904YLJ-VSV4h-02.ipc,13878,27,"[index, scan, header, rt, frag_type, collision...",2792,1908,20.12,13.75
1,PXD026629,20180904YLJ-VSV0h-03.ipc,14151,27,"[index, scan, header, rt, frag_type, collision...",3025,1702,21.38,12.03
2,PXD026629,20180831YLJ-H6h-01.ipc,13300,27,"[index, scan, header, rt, frag_type, collision...",2126,1014,15.98,7.62
3,PXD026629,20180831YLJ-H6h-02.ipc,13363,27,"[index, scan, header, rt, frag_type, collision...",2341,996,17.52,7.45
4,PXD026629,YangLuJie-LPS4h-2.ipc,12159,27,"[index, scan, header, rt, frag_type, collision...",2338,1444,19.23,11.88
5,PXD026629,20180831YLJ-H4h-02.ipc,14832,27,"[index, scan, header, rt, frag_type, collision...",2468,1096,16.64,7.39
6,PXD026629,20180904YLJ-VSV6h-01.ipc,14598,27,"[index, scan, header, rt, frag_type, collision...",3332,2067,22.83,14.16
7,PXD026629,YangLuJie-LPS8h-2.ipc,13334,27,"[index, scan, header, rt, frag_type, collision...",2329,1700,17.47,12.75
8,PXD026629,20180831YLJ-H0h-01.ipc,14194,27,"[index, scan, header, rt, frag_type, collision...",2361,1359,16.63,9.57
9,PXD026629,YangLuJie-M0-3.ipc,15028,27,"[index, scan, header, rt, frag_type, collision...",2593,1957,17.25,13.02


## PTMs identification

The scripts `scripts/identify_ptms` helps in identifying ptms.